# Layer test: Sentinel-1 SAR (VV / VH)

Ingest and display the **Sentinel-1** input layer. Filters and speckle match `761_Lecture6_Exercises.ipynb`.

This notebook does **not** train Random Forest. It only checks that GRD scenes load, clip, and draw on a map.


In [ ]:
import sys
from pathlib import Path

import ee
import geemap

_root = Path.cwd()
_nb = _root / "notebooks" if (_root / "notebooks" / "layer_config.py").exists() else _root
sys.path.insert(0, str(_nb))
import layer_config as cfg

ee.Initialize(project=cfg.GEE_PROJECT)

aoi = ee.Geometry.Rectangle(cfg.AOI_BOUNDS)
print("GEE project:", cfg.GEE_PROJECT)
print("AOI:", cfg.AOI_BOUNDS)
print("Dates:", cfg.START_DATE, "→", cfg.END_DATE)

In [ ]:
s1_col = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filterDate(cfg.START_DATE, cfg.END_DATE)
    .filter(ee.Filter.eq("instrumentMode", "IW"))
    .filter(ee.Filter.eq("orbitProperties_pass", "DESCENDING"))
    .filter(ee.Filter.eq("resolution_meters", 10))
    .select(["VV", "VH"])
)

n = s1_col.size().getInfo()
print("S1 scenes after filters:", n)
assert n > 0, "No Sentinel-1 scenes. Widen START_DATE / END_DATE in layer_config.py."

info = s1_col.first().getInfo()
print("First scene id:", info["id"])
print("Bands:", [b["id"] for b in info["bands"]])

In [ ]:
s1 = s1_col.mean().clip(aoi)
s1_speckle = s1.focal_mean(radius=50, units="meters")

vv_vis = {"min": -20, "max": 0}
Map = geemap.Map(center=cfg.MAP_CENTER, zoom=cfg.MAP_ZOOM, basemap="HYBRID")
Map.addLayer(s1.select("VV"), vv_vis, "S1 VV mean")
Map.addLayer(s1.select("VH"), vv_vis, "S1 VH mean")
Map.addLayer(s1_speckle.select("VV"), vv_vis, "S1 VV speckle 50 m")
Map.addLayer(aoi, {"color": "red"}, "AOI")
Map